# 🚢 Agentic AI for Maritime Freight Pricing and Route Optimization
## Milestone 1 · User Authentication Module
**Infosys Springboard 7.0 · Batch 4** — Streamlit + JWT + SQLite + ngrok + Gmail OTP

**Features:** Home page · Signup · Login (JWT session) · Forgot Password (Security Question + Email OTP) · User Dashboard with Settings · Admin Dashboard (user list + signup chart)

### How to run (top to bottom)
1. Add these **Colab Secrets** (🔑 left sidebar, *Notebook access* ON): `JWT_SECRET` (32+ random characters), `NGROK_AUTHTOKEN`, `EMAIL_ADDRESS`, `EMAIL_PASSWORD` (Gmail App Password), `ADMIN_USERNAME`, `ADMIN_PASSWORD`.
2. Run cells 1 → 2 → 3 → 4 → 5 in order. (Optional: create an `images` folder and upload pictures for the home page.)
3. The **last cell** starts the website and prints the URL. **Keep that cell running** while you use the site. Press ■ Stop on that cell to shut the site down.
4. After a runtime restart, run cells 1 → 5 again, then the last cell.

**Theme:** `classic` by default. Change `THEME = "classic"` to `"ocean"` inside the app cell for the blue theme.

In [ ]:
!pip install -q streamlit pyngrok PyJWT bcrypt

In [ ]:
import os
from google.colab import userdata

SECRET_NAMES = ["JWT_SECRET", "NGROK_AUTHTOKEN", "EMAIL_ADDRESS",
                "EMAIL_PASSWORD", "ADMIN_USERNAME", "ADMIN_PASSWORD"]

missing = []
for name in SECRET_NAMES:
    try:
        os.environ[name] = userdata.get(name).strip()
    except Exception:
        missing.append(name)

print("Missing secrets:", missing if missing else "none")


In [ ]:
import os
os.makedirs(".streamlit", exist_ok=True)
os.makedirs("images", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#0e7490"\nbackgroundColor="#f7fbfc"\nsecondaryBackgroundColor="#e6f4f6"\ntextColor="#0b2545"\n')
print("Theme configured. Upload your pictures into the 'images' folder (Files panel).")


In [ ]:
%%writefile app.py
import glob
import hmac
import os
import re
import secrets
import smtplib
import sqlite3
import time
from contextlib import closing
from datetime import datetime, timedelta, timezone
from email.message import EmailMessage
from html import escape

import base64
import bcrypt
import jwt
import pandas as pd
import streamlit as st

APP_TITLE = "Agentic AI for Maritime Freight Pricing and Route Optimization"
SHORT_TITLE = "Maritime Freight AI"

st.set_page_config(page_title=SHORT_TITLE, page_icon="🚢", layout="wide")


# Config (all secrets come from environment variables set in Colab)

JWT_SECRET = os.environ.get("JWT_SECRET", "")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "").replace(" ", "")  # Gmail shows the app password with spaces
ADMIN_USERNAME = os.environ.get("ADMIN_USERNAME", "")
ADMIN_PASSWORD = os.environ.get("ADMIN_PASSWORD", "")

DB_FILE = "auth_users.db"
TOKEN_MINUTES = 30
OTP_MINUTES = 5
OTP_MAX_TRIES = 5

SECURITY_QUESTIONS = [
    "What is your pet's name?",
    "What is your mother's maiden name?",
    "What was the name of your first school?",
    "What is your favourite city?",
    "What is your favourite food?",
]

USER_MENU = ["🏠 Dashboard", "⚙️ Settings", "🚪 Logout"]
ADMIN_MENU = ["📊 Dashboard", "🚪 Logout"]


# Styling

STYLE_OCEAN = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@500;600;700&family=Inter:wght@400;500;600&display=swap');
html, body, .stApp { background: #f3f8fb; font-family: 'Inter', sans-serif; color: #16324f; }
h1, h2, h3, h4 { font-family: 'Poppins', sans-serif; color: #0a2342; }
header[data-testid="stHeader"] { background: transparent; }
.block-container { padding-top: 2rem; max-width: 1150px; }

.hero { background: linear-gradient(135deg, #0a2342 0%, #126782 60%, #2ca6a4 100%);
  border-radius: 20px; padding: 40px 36px 74px; position: relative; overflow: hidden;
  box-shadow: 0 10px 30px rgba(10, 35, 66, 0.25); margin-bottom: 22px; }
.hero h1 { color: #ffffff; font-size: 2.05rem; margin: 0 0 8px; line-height: 1.25; }
.hero p { color: #d6eef0; margin: 0; font-size: 1.02rem; }
.hero .badge { display: inline-block; background: rgba(255,255,255,0.15);
  border: 1px solid rgba(255,255,255,0.35); color: #ffffff; padding: 4px 14px;
  border-radius: 999px; font-size: 0.8rem; margin-bottom: 14px; }
.hero svg { position: absolute; left: 0; bottom: -1px; width: 100%; height: 46px; }

.auth-head { text-align: center; padding: 6px 0 14px; font-size: 38px; }
.auth-brand { font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 1.25rem; color: #0a2342; }
.auth-title { color: #126782; font-weight: 600; margin-top: 4px; font-size: 1rem; }

div[data-testid="stForm"] { background: #ffffff; border: 1px solid #d5e6ec; border-radius: 16px;
  padding: 22px; box-shadow: 0 6px 18px rgba(10, 35, 66, 0.08); }
div[data-testid="stButton"] button, div[data-testid="stFormSubmitButton"] button {
  background: #126782; color: #ffffff; border: none; border-radius: 10px;
  font-weight: 600; padding: 0.55rem 1rem; width: 100%; }
div[data-testid="stButton"] button p, div[data-testid="stFormSubmitButton"] button p { color: #ffffff; }
div[data-testid="stButton"] button:hover, div[data-testid="stFormSubmitButton"] button:hover {
  background: #0a2342; color: #ffffff; }

.metric { background: #ffffff; border: 1px solid #d5e6ec; border-radius: 14px; padding: 16px 18px;
  box-shadow: 0 4px 12px rgba(10, 35, 66, 0.06); min-height: 108px; }
.m-label { color: #5b7a8c; font-size: 0.8rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.04em; }
.m-value { color: #0a2342; font-family: 'Poppins', sans-serif; font-size: 1.35rem; font-weight: 700; margin-top: 4px; word-break: break-word; }
.m-note { color: #5b7a8c; font-size: 0.8rem; margin-top: 2px; }

.feature { background: #ffffff; border: 1px solid #d5e6ec; border-radius: 16px; padding: 20px;
  box-shadow: 0 4px 12px rgba(10, 35, 66, 0.06); min-height: 190px; }
.f-icon { font-size: 30px; }
.f-title { font-family: 'Poppins', sans-serif; font-weight: 600; color: #0a2342; margin: 6px 0 4px; }
.f-text { color: #3d5a6c; font-size: 0.92rem; margin-bottom: 12px; }
.chip { display: inline-block; border-radius: 999px; padding: 3px 12px; font-size: 0.75rem; font-weight: 600; }
.chip-live { background: #d9f5ea; color: #0b7a4b; }
.chip-soon { background: #fff1d6; color: #9a6200; }

section[data-testid="stSidebar"] { background: #0a2342; }
section[data-testid="stSidebar"] * { color: #e6f1f7 !important; }
.side-brand { text-align: center; font-size: 34px; padding-top: 6px; }
.side-title { font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 1rem; }
.side-sub { font-size: 0.75rem; opacity: 0.8; }
.side-user { background: rgba(255,255,255,0.1); border-radius: 10px; padding: 8px 12px; margin: 12px 0; font-size: 0.9rem; word-break: break-word; }
</style>
"""

STYLE_CLASSIC = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@500;600;700&family=Inter:wght@400;500;600&display=swap');
html, body, .stApp { background: #f9fcfc; font-family: 'Inter', sans-serif; color: #2d334a; }
h1, h2, h3, h4 { font-family: 'Poppins', sans-serif; color: #272343; }
header[data-testid="stHeader"] { background: transparent; }
.block-container { padding-top: 2rem; max-width: 1150px; }

.hero { background: #272343; border: 2px solid #272343; border-radius: 16px; padding: 38px 34px 72px;
  position: relative; overflow: hidden; box-shadow: 6px 6px 0px #bae8e8; margin-bottom: 22px; }
.hero h1 { color: #ffd803; font-size: 2.05rem; margin: 0 0 8px; line-height: 1.25; }
.hero p { color: #bae8e8; margin: 0; font-size: 1.02rem; }
.hero .badge { display: inline-block; background: #ffd803; color: #272343; padding: 4px 14px;
  border-radius: 999px; font-size: 0.8rem; font-weight: 700; margin-bottom: 14px; }
.hero svg { position: absolute; left: 0; bottom: -1px; width: 100%; height: 46px; }

.auth-head { text-align: center; padding: 6px 0 14px; font-size: 38px; }
.auth-brand { font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 1.25rem; color: #272343; }
.auth-title { color: #272343; font-weight: 700; margin-top: 4px; font-size: 1rem; }

div[data-testid="stForm"] { background: #ffffff; border: 2px solid #272343; border-radius: 16px;
  padding: 22px; box-shadow: 6px 6px 0px #bae8e8; }
div[data-baseweb="base-input"] { background-color: transparent !important; border: none !important; }
div[data-baseweb="input"] { background-color: #ffffff !important; border: 2px solid #272343 !important; border-radius: 10px !important; }
div[data-baseweb="input"]:focus-within { border-color: #ffd803 !important; box-shadow: 4px 4px 0px #272343 !important; }
div[data-baseweb="select"] > div { background-color: #ffffff !important; border: 2px solid #272343 !important; border-radius: 10px !important; }
div[data-testid="stButton"] button, div[data-testid="stFormSubmitButton"] button {
  background: #ffd803; color: #272343; border: 2px solid #272343; border-radius: 10px;
  font-weight: 700; padding: 0.55rem 1rem; width: 100%; box-shadow: 4px 4px 0px #272343; transition: all 0.2s ease; }
div[data-testid="stButton"] button p, div[data-testid="stFormSubmitButton"] button p { color: #272343; }
div[data-testid="stButton"] button:hover, div[data-testid="stFormSubmitButton"] button:hover {
  background: #e6c300; color: #272343; transform: translate(-2px, -2px); box-shadow: 6px 6px 0px #272343; }

.metric { background: #ffffff; border: 2px solid #272343; border-radius: 14px; padding: 16px 18px;
  box-shadow: 4px 4px 0px #bae8e8; min-height: 108px; }
.m-label { color: #64748b; font-size: 0.8rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.04em; }
.m-value { color: #272343; font-family: 'Poppins', sans-serif; font-size: 1.35rem; font-weight: 700; margin-top: 4px; word-break: break-word; }
.m-note { color: #64748b; font-size: 0.8rem; margin-top: 2px; }

.feature { background: #ffffff; border: 2px solid #272343; border-radius: 16px; padding: 20px;
  box-shadow: 4px 4px 0px #bae8e8; min-height: 190px; }
.f-icon { font-size: 30px; }
.f-title { font-family: 'Poppins', sans-serif; font-weight: 600; color: #272343; margin: 6px 0 4px; }
.f-text { color: #2d334a; font-size: 0.92rem; margin-bottom: 12px; }
.chip { display: inline-block; border: 1.5px solid #272343; border-radius: 999px; padding: 3px 12px; font-size: 0.75rem; font-weight: 700; color: #272343; }
.chip-live { background: #bae8e8; }
.chip-soon { background: #ffd803; }

section[data-testid="stSidebar"] { background: #e3f6f5; border-right: 2px solid #272343; }
section[data-testid="stSidebar"] * { color: #272343 !important; }
.side-brand { text-align: center; font-size: 34px; padding-top: 6px; }
.side-title { font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 1rem; }
.side-sub { font-size: 0.75rem; opacity: 0.8; }
.side-user { background: #ffffff; border: 2px solid #272343; border-radius: 10px; padding: 8px 12px; margin: 12px 0; font-size: 0.9rem; word-break: break-word; }
</style>
"""

# Theme: "harbor" (navy + teal + coral with sea background), "classic" (yellow + navy) or "ocean" (blue + teal)
THEME = "harbor"
STYLE = STYLE_CLASSIC if THEME in ("classic", "harbor") else STYLE_OCEAN
WAVE_FILL = "#f9fcfc" if THEME == "classic" else "#f3f8fb"

WAVE = (
    '<svg viewBox="0 0 1200 60" preserveAspectRatio="none">'
    '<path d="M0,30 C150,60 350,0 600,30 C850,60 1050,0 1200,30 L1200,60 L0,60 Z" fill="' + WAVE_FILL + '"/></svg>'
)

STYLE_EXTRA = """
<style>
.ship { display: inline-block; animation: floaty 3s ease-in-out infinite; }
@keyframes floaty { 0%, 100% { transform: translateY(0); } 50% { transform: translateY(-6px); } }
.stat { background: #ffffff; border: 2px solid __INK__; border-radius: 14px; padding: 14px 16px;
  text-align: center; box-shadow: 4px 4px 0px __SOFT__; }
.stat b { display: block; font-family: 'Poppins', sans-serif; font-size: 1.5rem; color: __INK__; }
.stat span { font-size: 0.75rem; color: #64748b; font-weight: 600; text-transform: uppercase; letter-spacing: 0.04em; }
.tech { display: inline-block; background: __SOFT__; border: 1.5px solid __INK__; border-radius: 999px;
  padding: 4px 14px; margin: 4px 6px 4px 0; font-size: 0.82rem; font-weight: 600; color: __INK__; }
.site-footer { text-align: center; color: #64748b; font-size: 0.8rem; margin-top: 28px;
  padding-top: 14px; border-top: 1px solid __SOFT__; }
</style>
"""
_INK, _SOFT = ("#272343", "#bae8e8") if THEME == "classic" else ("#0a2342", "#d5e6ec")
STYLE_EXTRA = STYLE_EXTRA.replace("__INK__", _INK).replace("__SOFT__", _SOFT)

HARBOR_SVG = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1600 900" preserveAspectRatio="xMidYMid slice">
<defs>
<linearGradient id="sky" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#040d20"/><stop offset="0.5" stop-color="#0f3d62"/><stop offset="1" stop-color="#1b8aa0"/></linearGradient>
<linearGradient id="sea" x1="0" y1="0" x2="0" y2="1"><stop offset="0" stop-color="#127a93"/><stop offset="1" stop-color="#05253f"/></linearGradient>
<radialGradient id="glow"><stop offset="0" stop-color="#fff4d6" stop-opacity="0.55"/><stop offset="1" stop-color="#fff4d6" stop-opacity="0"/></radialGradient>
</defs>
<rect width="1600" height="900" fill="url(#sky)"/>
<g fill="#ffffff" opacity="0.75"><circle cx="120" cy="90" r="2"/><circle cx="330" cy="160" r="1.5"/><circle cx="560" cy="60" r="2"/><circle cx="800" cy="130" r="1.5"/><circle cx="980" cy="70" r="2"/><circle cx="1150" cy="190" r="1.5"/><circle cx="1480" cy="90" r="2"/><circle cx="70" cy="250" r="1.5"/><circle cx="660" cy="250" r="1.5"/></g>
<circle cx="1300" cy="190" r="170" fill="url(#glow)"/><circle cx="1300" cy="190" r="52" fill="#fff4d6"/>
<path d="M0,600 L120,560 L260,590 L420,540 L560,585 L700,560 L900,600 L1100,550 L1300,590 L1450,555 L1600,590 L1600,640 L0,640Z" fill="#0a2a47" opacity="0.85"/>
<rect y="600" width="1600" height="300" fill="url(#sea)"/>
<g transform="translate(120,0)">
<path d="M150,605 L760,605 L720,660 L190,660 Z" fill="#06142a"/>
<rect x="190" y="560" width="70" height="45" fill="#ff7a59"/><rect x="260" y="560" width="70" height="45" fill="#f5b700"/><rect x="330" y="560" width="70" height="45" fill="#13a8a8"/><rect x="400" y="560" width="70" height="45" fill="#cbd5e1"/><rect x="470" y="560" width="70" height="45" fill="#ff7a59"/>
<rect x="215" y="515" width="70" height="45" fill="#13a8a8"/><rect x="285" y="515" width="70" height="45" fill="#cbd5e1"/><rect x="355" y="515" width="70" height="45" fill="#f5b700"/>
<rect x="620" y="500" width="70" height="105" fill="#0b2545"/><rect x="610" y="490" width="90" height="16" fill="#06142a"/>
<g fill="#ffd98a"><rect x="630" y="512" width="10" height="8"/><rect x="650" y="512" width="10" height="8"/><rect x="670" y="512" width="10" height="8"/></g>
<rect x="654" y="450" width="4" height="40" fill="#06142a"/>
</g>
<path d="M0,690 C200,650 420,730 660,690 S1100,650 1320,700 S1520,680 1600,690 L1600,900 L0,900Z" fill="#0e7490" opacity="0.55"/>
<path d="M0,760 C240,720 480,800 760,750 S1200,720 1600,770 L1600,900 L0,900Z" fill="#075985" opacity="0.7"/>
<path d="M0,820 C300,790 560,860 900,820 S1400,790 1600,830 L1600,900 L0,900Z" fill="#04263f" opacity="0.9"/>
</svg>"""


@st.cache_data
def _background_css():
    for ext in ("jpg", "jpeg", "png", "webp"):
        for path in glob.glob("images/background." + ext) + glob.glob("images/background." + ext.upper()):
            mime = "jpeg" if ext in ("jpg", "jpeg") else ext
            data = base64.b64encode(open(path, "rb").read()).decode()
            return ("linear-gradient(rgba(4,16,36,0.45), rgba(4,16,36,0.45)), "
                    f"url(data:image/{mime};base64,{data}) center / cover no-repeat fixed")
    data = base64.b64encode(HARBOR_SVG.encode()).decode()
    return f"url(data:image/svg+xml;base64,{data}) center / cover no-repeat fixed"


STYLE_HARBOR = """
<style>
html, body, .stApp { background: __BG__ !important; }
[data-testid="stAppViewContainer"], [data-testid="stMain"] { background: transparent !important; }
header[data-testid="stHeader"] { background: transparent !important; }
.block-container { background: #f7fbfc; border: 1px solid rgba(255,255,255,0.7); border-radius: 24px;
  box-shadow: 0 24px 70px rgba(2, 12, 27, 0.55); padding: 2.2rem 2.4rem 2rem !important;
  margin-top: 1.5rem; margin-bottom: 2rem; }
h1, h2, h3, h4, .auth-brand, .auth-title { color: #0b2545; }
.hero { background: linear-gradient(135deg, #0b2545 0%, #13506b 55%, #0e8a99 100%); border: none;
  border-radius: 20px; box-shadow: 0 14px 34px rgba(11, 37, 69, 0.35); }
.hero h1 { color: #ffffff; }
.hero p { color: #cdeff2; }
.hero .badge { background: #ff7a59; color: #0b2545; }
.hero svg path { fill: #f7fbfc !important; }
div[data-testid="stForm"] { background: #ffffff; border: 1px solid #cfe3ea; border-top: 4px solid #0e7490;
  border-radius: 18px; box-shadow: 0 10px 28px rgba(11, 37, 69, 0.12); }
div[data-baseweb="input"], div[data-baseweb="select"] > div { border: 1.5px solid #9cc3cf !important; border-radius: 10px !important; }
div[data-baseweb="input"]:focus-within { border-color: #0e7490 !important; box-shadow: 0 0 0 3px rgba(14, 116, 144, 0.22) !important; }
div[data-testid="stButton"] button, div[data-testid="stFormSubmitButton"] button {
  background: linear-gradient(135deg, #0e7490, #155e75); color: #ffffff; border: none; border-radius: 12px;
  box-shadow: 0 6px 16px rgba(14, 116, 144, 0.35); }
div[data-testid="stButton"] button p, div[data-testid="stFormSubmitButton"] button p { color: #ffffff !important; }
div[data-testid="stButton"] button:hover, div[data-testid="stFormSubmitButton"] button:hover {
  background: linear-gradient(135deg, #ff7a59, #f0603c); color: #ffffff; transform: translateY(-2px);
  box-shadow: 0 10px 22px rgba(255, 122, 89, 0.45); }
.metric, .feature, .stat { background: #ffffff; border: 1px solid #cfe3ea; border-top: 4px solid #0e7490;
  border-radius: 16px; box-shadow: 0 8px 22px rgba(11, 37, 69, 0.10); }
.chip { border-color: #0e7490; color: #0b2545; }
.chip-live { background: #d9f3f5; }
.chip-soon { background: #ffe1d8; }
section[data-testid="stSidebar"] { background: linear-gradient(180deg, #0b2545, #13506b); border-right: none; }
section[data-testid="stSidebar"] * { color: #ffffff !important; }
.side-user { background: rgba(255, 255, 255, 0.12); border: 1px solid rgba(255, 255, 255, 0.25); }
@media (max-width: 640px) { .block-container { padding: 1.2rem 1rem !important; border-radius: 16px; margin-top: 0.5rem; } }
</style>
"""
if THEME == "harbor":
    STYLE_EXTRA += STYLE_HARBOR.replace("__BG__", _background_css())

st.markdown(STYLE + STYLE_EXTRA, unsafe_allow_html=True)

if not JWT_SECRET:
    st.error("JWT_SECRET is missing. Add it in Colab Secrets and re-run the notebook.")
    st.stop()


def render_html(markup):
    cleaned = "\n".join(line.strip() for line in markup.strip().splitlines() if line.strip())
    st.markdown(cleaned, unsafe_allow_html=True)


def hero(badge="", title="", subtitle=""):
    badge_html = f'<span class="badge">{badge}</span>' if badge else ""
    render_html(
        f'<div class="hero">{badge_html}<h1>{title}</h1><p>{subtitle}</p>{WAVE}</div>'
    )


def metric_card(label, value, note=""):
    return (
        f'<div class="metric"><div class="m-label">{label}</div>'
        f'<div class="m-value">{value}</div><div class="m-note">{note}</div></div>'
    )


def show_metrics(items):
    cols = st.columns(len(items))
    for col, (label, value, note) in zip(cols, items):
        with col:
            render_html(metric_card(label, value, note))


def feature_card(icon, title, text, chip, kind):
    return (
        f'<div class="feature"><div class="f-icon">{icon}</div><div class="f-title">{title}</div>'
        f'<div class="f-text">{text}</div><span class="chip {kind}">{chip}</span></div>'
    )


def roadmap():
    st.subheader("Project roadmap")
    c1, c2, c3 = st.columns(3)
    with c1:
        render_html(feature_card(
            "🔐", "Secure Access",
            "Secure signup and login, password recovery by security question or email code, and an admin panel.",
            "Live in Milestone 1", "chip-live"))
    with c2:
        render_html(feature_card(
            "💰", "Freight Pricing Agent",
            "AI-assisted estimation of ocean freight rates.",
            "Planned", "chip-soon"))
    with c3:
        render_html(feature_card(
            "🧭", "Route Optimization",
            "Smarter route planning across shipping lanes.",
            "Planned", "chip-soon"))


def show_stats(items):
    cols = st.columns(len(items))
    for col, (value, label) in zip(cols, items):
        with col:
            render_html(f'<div class="stat"><b>{value}</b><span>{label}</span></div>')


def tech_stack(items):
    render_html("<div>" + "".join(f'<span class="tech">{t}</span>' for t in items) + "</div>")


def site_footer():
    render_html(
        f'<div class="site-footer">{APP_TITLE}<br>'
        "Infosys Springboard 7.0 · Batch 4 · Milestone 1 — User Authentication Module</div>"
    )


def centered():
    return st.columns([1, 1.6, 1])[1]


def auth_header(title):
    render_html(
        f'<div class="auth-head">🚢<div class="auth-brand">{SHORT_TITLE}</div>'
        f'<div class="auth-title">{title}</div></div>'
    )



# Database

def run_sql(sql, params=(), fetch=None, commit=False):
    with closing(sqlite3.connect(DB_FILE)) as conn:
        conn.execute(
            """CREATE TABLE IF NOT EXISTS users(
                username TEXT PRIMARY KEY COLLATE NOCASE,
                email TEXT UNIQUE,
                password_hash BLOB,
                sec_question TEXT,
                sec_answer_hash BLOB,
                created_at TEXT)"""
        )
        cur = conn.execute(sql, params)
        result = None
        if fetch == "one":
            result = cur.fetchone()
        elif fetch == "all":
            result = cur.fetchall()
        if commit:
            conn.commit()
        return result



# Hashing, validation, JWT, email
def hash_text(text):
    return bcrypt.hashpw(text.encode(), bcrypt.gensalt())


def check_text(text, hashed):
    try:
        return bcrypt.checkpw(text.encode(), hashed)
    except Exception:
        return False


EMAIL_RE = re.compile(
    r"(?=(?:[^@]*[A-Za-z]){2})[A-Za-z0-9._%+-]+@[A-Za-z]{2,}(?:\.[A-Za-z]{2,})*\.[A-Za-z]{2,}"
)


def valid_email(email):
    return EMAIL_RE.fullmatch(email) is not None


def password_errors(pw, confirm):
    errors = []
    if len(pw) < 8:
        errors.append("at least 8 characters")
    if not re.search(r"[A-Z]", pw):
        errors.append("one uppercase letter")
    if not re.search(r"[a-z]", pw):
        errors.append("one lowercase letter")
    if not re.search(r"\d", pw):
        errors.append("one number")
    if not re.search(r"[^A-Za-z0-9]", pw):
        errors.append("one special symbol")
    msg = ""
    if errors:
        msg = "Password needs: " + ", ".join(errors) + "."
    elif pw != confirm:
        msg = "Password and Confirm Password do not match."
    return msg


def make_token(subject, role="user"):
    payload = {
        "sub": subject,
        "role": role,
        "exp": datetime.now(timezone.utc) + timedelta(minutes=TOKEN_MINUTES),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def token_payload(token, role="user"):
    if not token:
        return None
    try:
        data = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
    except Exception:
        return None
    return data if data.get("role") == role else None


def minutes_left(payload):
    return max(0, int((payload["exp"] - time.time()) // 60))


def send_otp_email(to_email, otp):
    msg = EmailMessage()
    msg["Subject"] = f"Your OTP for {SHORT_TITLE}"
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = to_email
    msg.set_content(
        f"Your one-time code is {otp}.\n"
        f"It is valid for {OTP_MINUTES} minutes.\n"
        "If you did not request this, please ignore this email."
    )
    with smtplib.SMTP_SSL("smtp.gmail.com", 465, timeout=20) as server:
        server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
        server.send_message(msg)



# Navigation helpers
if "page" not in st.session_state:
    st.session_state.page = "home"


def go(page):
    st.session_state.page = page
    st.rerun()


def logout():
    st.session_state.clear()
    st.rerun()


def clear_forgot_state():
    for key in list(st.session_state.keys()):
        if key.startswith("fp_"):
            del st.session_state[key]


def flash(kind, text):
    st.session_state["flash"] = (kind, text)


def show_flash():
    item = st.session_state.pop("flash", None)
    if item:
        getattr(st, item[0])(item[1])



# Public pages

def page_home():
    hero(
        badge="",
        title=f'<span class="ship">🚢</span> {APP_TITLE}',
        subtitle="A secure, AI-ready platform for smarter ocean freight pricing and route planning. "
        "Sign in to get started.",
    )
    files = []
    for ext in ("jpg", "jpeg", "png", "webp"):
        files += glob.glob("images/*." + ext) + glob.glob("images/*." + ext.upper())
    files = [f for f in sorted(set(files)) if not os.path.basename(f).lower().startswith("background")]
    for start in range(0, len(files), 3):
        row = files[start:start + 3]
        cols = st.columns(len(row))
        for col, path in zip(cols, row):
            col.image(path)
    c1, c2, c3 = st.columns(3)
    if c1.button("Login", key="home_login"):
        go("login")
    if c2.button("Sign Up", key="home_signup"):
        go("signup")
    if c3.button("Admin Login", key="home_admin"):
        go("admin_login")




def page_login():
    with centered():
        auth_header("Sign in to your account")
        show_flash()
        with st.form("login_form"):
            ident = st.text_input("Username or Email", key="login_ident").strip()
            pw = st.text_input("Password", type="password", key="login_pw")
            submit = st.form_submit_button("Login")
        if submit:
            if not ident or not pw:
                st.error("All fields are required.")
            else:
                row = run_sql(
                    "SELECT username, password_hash FROM users WHERE username = ? OR email = ?",
                    (ident, ident.lower()),
                    fetch="one",
                )
                if row and check_text(pw, row[1]):
                    st.session_state["token"] = make_token(row[0])
                    go("dashboard")
                else:
                    st.error("Invalid credentials.")
        c1, c2 = st.columns(2)
        if c1.button("Create account", key="lg_signup"):
            go("signup")
        if c2.button("Forgot password", key="lg_forgot"):
            clear_forgot_state()
            go("forgot")
        c3, c4 = st.columns(2)
        if c3.button("Admin login", key="lg_admin"):
            go("admin_login")
        if c4.button("← Home", key="lg_home"):
            go("home")


def page_signup():
    with centered():
        auth_header("Create your account")
        with st.form("signup_form"):
            username = st.text_input("Username", key="su_user").strip()
            email = st.text_input("Email", key="su_email").strip().lower()
            pw = st.text_input("Password", type="password", key="su_pw")
            confirm = st.text_input("Confirm Password", type="password", key="su_confirm")
            question = st.selectbox("Security Question", SECURITY_QUESTIONS, key="su_q")
            answer = st.text_input("Security Answer", key="su_a").strip()
            submit = st.form_submit_button("Create Account")
        if submit:
            if not all([username, email, pw, confirm, answer]):
                st.error("All fields are required.")
            elif not valid_email(email):
                st.error("Invalid email. Use a format like ab@cd.ef (2+ letters in each part).")
            elif password_errors(pw, confirm):
                st.error(password_errors(pw, confirm))
            elif run_sql("SELECT 1 FROM users WHERE username = ?", (username,), fetch="one"):
                st.error("Username already exists. Please choose another one.")
            elif run_sql("SELECT 1 FROM users WHERE email = ?", (email,), fetch="one"):
                st.error("This email is already registered.")
            else:
                run_sql(
                    "INSERT INTO users VALUES (?, ?, ?, ?, ?, ?)",
                    (
                        username,
                        email,
                        hash_text(pw),
                        question,
                        hash_text(answer.lower()),
                        datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    ),
                    commit=True,
                )
                flash("success", "Account created. Please login.")
                go("login")
        if st.button("← Back to Login", key="su_back"):
            go("login")


def finish_reset():
    flash("success", "Password reset successful. Please login with your new password.")
    clear_forgot_state()
    st.session_state.pop("token", None)
    go("login")


def new_password_form(form_key, extra_fields=None):
    """Shared 'set new password' form. extra_fields: callable rendering extra inputs."""
    with st.form(form_key):
        extra = extra_fields() if extra_fields else None
        new_pw = st.text_input("New Password", type="password", key=form_key + "_pw")
        confirm = st.text_input("Confirm New Password", type="password", key=form_key + "_cf")
        submit = st.form_submit_button("Reset Password")
    return submit, extra, new_pw, confirm


def update_password(username, new_pw):
    run_sql(
        "UPDATE users SET password_hash = ? WHERE username = ?",
        (hash_text(new_pw), username),
        commit=True,
    )


def forgot_security_question():
    user = st.session_state.get("fp_q_user")
    if not user:
        with st.form("fp_q_user_form"):
            uname = st.text_input("Username", key="fp_q_uname").strip()
            ok = st.form_submit_button("Continue")
        if ok:
            if not uname:
                st.error("Username is required.")
            else:
                row = run_sql(
                    "SELECT username, sec_question FROM users WHERE username = ?",
                    (uname,),
                    fetch="one",
                )
                if row:
                    st.session_state["fp_q_user"] = row[0]
                    st.session_state["fp_q_text"] = row[1]
                    st.rerun()
                else:
                    st.error("Username not found.")
        return

    st.info(f"Security question: {st.session_state['fp_q_text']}")

    def answer_box():
        return st.text_input("Your Answer", key="fp_q_answer").strip()

    submit, answer, new_pw, confirm = new_password_form("fp_q_reset", answer_box)
    if submit:
        if not answer or not new_pw or not confirm:
            st.error("All fields are required.")
        elif password_errors(new_pw, confirm):
            st.error(password_errors(new_pw, confirm))
        else:
            row = run_sql(
                "SELECT sec_answer_hash FROM users WHERE username = ?", (user,), fetch="one"
            )
            if row and check_text(answer.lower(), row[0]):
                update_password(user, new_pw)
                finish_reset()
            else:
                st.error("Incorrect answer.")


def forgot_otp():
    stage = st.session_state.get("fp_otp_stage", "email")

    if stage == "email":
        with st.form("fp_otp_email_form"):
            email = st.text_input("Registered Email", key="fp_otp_email").strip().lower()
            ok = st.form_submit_button("Send OTP")
        if ok:
            if not email:
                st.error("Email is required.")
            elif not valid_email(email):
                st.error("Invalid email format.")
            elif not (EMAIL_ADDRESS and EMAIL_PASSWORD):
                st.error("Email service is not configured.")
            else:
                row = run_sql(
                    "SELECT username FROM users WHERE email = ?", (email,), fetch="one"
                )
                if not row:
                    st.error("This email is not registered.")
                else:
                    otp = f"{secrets.randbelow(900000) + 100000}"
                    try:
                        send_otp_email(email, otp)
                    except Exception:
                        st.error("Could not send the email. Check the Gmail secrets and try again.")
                    else:
                        st.session_state["fp_otp_user"] = row[0]
                        st.session_state["fp_otp_email_addr"] = email
                        st.session_state["fp_otp_code"] = otp
                        st.session_state["fp_otp_expiry"] = time.time() + OTP_MINUTES * 60
                        st.session_state["fp_otp_tries"] = 0
                        st.session_state["fp_otp_stage"] = "verify"
                        st.rerun()

    elif stage == "verify":
        st.info(f"OTP sent to {st.session_state['fp_otp_email_addr']}. Valid for {OTP_MINUTES} minutes.")
        with st.form("fp_otp_verify_form"):
            entered = st.text_input("Enter 6-digit OTP", max_chars=6, key="fp_otp_input").strip()
            ok = st.form_submit_button("Verify OTP")
        if ok:
            if not entered:
                st.error("OTP is required.")
            elif time.time() > st.session_state["fp_otp_expiry"]:
                st.error("OTP expired. Please request a new one.")
                st.session_state["fp_otp_stage"] = "email"
            elif st.session_state["fp_otp_tries"] >= OTP_MAX_TRIES:
                st.error("Too many wrong attempts. Please request a new OTP.")
                st.session_state["fp_otp_stage"] = "email"
            elif hmac.compare_digest(entered, st.session_state["fp_otp_code"]):
                st.session_state["fp_otp_stage"] = "reset"
                st.rerun()
            else:
                st.session_state["fp_otp_tries"] += 1
                st.error("Incorrect OTP.")

    elif stage == "reset":
        st.success("OTP verified. Set your new password.")
        user = st.session_state["fp_otp_user"]
        submit, _, new_pw, confirm = new_password_form("fp_otp_reset")
        if submit:
            if not new_pw or not confirm:
                st.error("All fields are required.")
            elif password_errors(new_pw, confirm):
                st.error(password_errors(new_pw, confirm))
            else:
                update_password(user, new_pw)
                finish_reset()


def page_forgot():
    with centered():
        auth_header("Reset your password")
        route = st.radio(
            "Choose recovery method",
            ["Security Question", "Email OTP"],
            horizontal=True,
            key="fp_route",
        )
        if route == "Security Question":
            forgot_security_question()
        else:
            forgot_otp()
        if st.button("← Back to Login", key="fp_back"):
            clear_forgot_state()
            go("login")


def page_admin_login():
    with centered():
        auth_header("Admin sign in")
        with st.form("admin_form"):
            name = st.text_input("Admin Username", key="ad_user").strip()
            pw = st.text_input("Admin Password", type="password", key="ad_pw")
            submit = st.form_submit_button("Login as Admin")
        if submit:
            if not name or not pw:
                st.error("All fields are required.")
            elif not (ADMIN_USERNAME and ADMIN_PASSWORD):
                st.error("Admin credentials are not configured.")
            elif hmac.compare_digest(name, ADMIN_USERNAME) and hmac.compare_digest(pw, ADMIN_PASSWORD):
                st.session_state["admin_token"] = make_token(name, role="admin")
                go("admin")
            else:
                st.error("Invalid credentials.")
        if st.button("← Back to Login", key="ad_back"):
            go("login")



# Logged-in user area

def sidebar_brand(role_label, name):
    render_html(
        f'<div class="side-brand">🚢<div class="side-title">{SHORT_TITLE}</div>'
        f'<div class="side-sub">{role_label}</div></div>'
    )
    render_html(f'<div class="side-user">👤 {escape(name)}</div>')


def user_dashboard(info, payload):
    username, email, question, created = info
    hero(
        "Signed in securely",
        f"Welcome aboard, {escape(username)} ⚓",
        "Your account is ready. Freight pricing and route optimization tools will appear here in upcoming milestones.",
    )
    show_metrics([
        ("Session", "Active ✅", "Your session is valid"),
        ("Session expires in", f"{minutes_left(payload)} min", "Login again after expiry"),
        ("Member since", escape((created or "-")[:10]), "Account creation date"),
        ("Recovery options", "2 enabled", "Security question + email OTP"),
    ])
    st.write("")
    roadmap()


def user_settings(info):
    username, email, question, created = info
    st.title("⚙️ Settings")
    tab_profile, tab_pw, tab_sq = st.tabs(["👤 Profile", "🔑 Change Password", "🛡️ Security Question"])

    with tab_profile:
        st.text_input("Username", value=username, disabled=True)
        st.text_input("Email", value=email, disabled=True)
        st.text_input("Member since", value=created or "-", disabled=True)
        st.text_input("Security question", value=question or "-", disabled=True)

    with tab_pw:
        with st.form("pw_change_form"):
            cur = st.text_input("Current Password", type="password", key="pc_cur")
            new = st.text_input("New Password", type="password", key="pc_new")
            conf = st.text_input("Confirm New Password", type="password", key="pc_conf")
            ok = st.form_submit_button("Update Password")
        if ok:
            if not all([cur, new, conf]):
                st.error("All fields are required.")
            else:
                row = run_sql(
                    "SELECT password_hash FROM users WHERE username = ?", (username,), fetch="one"
                )
                if not (row and check_text(cur, row[0])):
                    st.error("Current password is incorrect.")
                elif password_errors(new, conf):
                    st.error(password_errors(new, conf))
                elif new == cur:
                    st.error("New password must be different from the current password.")
                else:
                    update_password(username, new)
                    st.success("Password updated successfully.")

    with tab_sq:
        with st.form("sq_change_form"):
            new_q = st.selectbox("New Security Question", SECURITY_QUESTIONS, key="sq_new_q")
            new_a = st.text_input("New Security Answer", key="sq_new_a").strip()
            cur_pw = st.text_input("Current Password", type="password", key="sq_cur_pw")
            ok = st.form_submit_button("Update Security Question")
        if ok:
            if not all([new_a, cur_pw]):
                st.error("All fields are required.")
            else:
                row = run_sql(
                    "SELECT password_hash FROM users WHERE username = ?", (username,), fetch="one"
                )
                if not (row and check_text(cur_pw, row[0])):
                    st.error("Current password is incorrect.")
                else:
                    run_sql(
                        "UPDATE users SET sec_question = ?, sec_answer_hash = ? WHERE username = ?",
                        (new_q, hash_text(new_a.lower()), username),
                        commit=True,
                    )
                    st.success("Security question updated.")


def user_area():
    payload = token_payload(st.session_state.get("token"))
    if not payload:
        st.session_state.pop("token", None)
        flash("warning", "Session expired or invalid. Please login again.")
        go("login")
    username = payload["sub"]
    info = run_sql(
        "SELECT username, email, sec_question, created_at FROM users WHERE username = ?",
        (username,),
        fetch="one",
    )
    if not info:
        st.session_state.pop("token", None)
        go("login")
    with st.sidebar:
        sidebar_brand("User Portal", info[0])
        choice = st.radio("Menu", USER_MENU, key="menu_user", label_visibility="collapsed")
    if choice == "🚪 Logout":
        logout()
    elif choice == "⚙️ Settings":
        user_settings(info)
    else:
        user_dashboard(info, payload)



# Admin area
def admin_area():
    payload = token_payload(st.session_state.get("admin_token"), role="admin")
    if not payload:
        st.session_state.pop("admin_token", None)
        flash("warning", "Admin session expired. Please login again.")
        go("admin_login")
    admin = payload["sub"]
    with st.sidebar:
        sidebar_brand("Admin Panel", admin)
        choice = st.radio("Menu", ADMIN_MENU, key="menu_admin", label_visibility="collapsed")
    if choice == "🚪 Logout":
        logout()

    hero(
        "Admin Control Panel",
        f"Welcome, {escape(admin)} 🛡️",
        "Monitor registered users of the platform. Passwords are never displayed.",
    )
    rows = run_sql(
        "SELECT username, email, created_at FROM users ORDER BY created_at DESC", fetch="all"
    )
    today = datetime.now().strftime("%Y-%m-%d")
    joined_today = sum(1 for r in rows if (r[2] or "").startswith(today))
    show_metrics([
        ("Total users", str(len(rows)), "Registered accounts"),
        ("Joined today", str(joined_today), "New signups today"),
        ("Latest signup", escape(rows[0][0]) if rows else "-", "Most recent user"),
        ("Session expires in", f"{minutes_left(payload)} min", "Admin session"),
    ])
    st.write("")
    st.subheader("Signups per day")
    per_day = {}
    for r in rows:
        day = (r[2] or "")[:10]
        if day:
            per_day[day] = per_day.get(day, 0) + 1
    if per_day:
        st.bar_chart(pd.DataFrame({"Signups": dict(sorted(per_day.items()))}))
    else:
        st.info("No signups yet.")
    st.subheader("Registered users")
    query = st.text_input("Search", placeholder="Search by username or email", key="adm_search").strip().lower()
    shown = [r for r in rows if query in r[0].lower() or query in (r[1] or "").lower()] if query else rows
    if shown:
        st.dataframe(
            [{"Username": r[0], "Email": r[1], "Created At": r[2]} for r in shown],
            hide_index=True,
        )
    else:
        st.info("No users found.")



# Router

PAGES = {
    "home": page_home,
    "login": page_login,
    "signup": page_signup,
    "forgot": page_forgot,
    "admin_login": page_admin_login,
}

current = st.session_state.page
if current == "dashboard":
    user_area()
elif current == "admin":
    admin_area()
else:
    PAGES.get(current, page_home)()




In [ ]:
import os
RESET_DB = False

if RESET_DB:
    if os.path.exists("auth_users.db"):
        os.remove("auth_users.db")
    print("Database reset.")
else:
    print("Database kept (RESET_DB is False).")

In [ ]:
import os, re, time, socket, subprocess
from google.colab import userdata
from pyngrok import ngrok

if not os.path.exists("app.py"):
    raise SystemExit("app.py not found. Run the app cell (cell 4) first.")
if not os.environ.get("JWT_SECRET"):
    raise SystemExit("Secrets not loaded. Run the secrets cell (cell 2) first.")


ngrok.kill()
!pkill -f streamlit
time.sleep(2)


log = open("st.log", "w")
server = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    env=os.environ.copy(), stdout=log, stderr=subprocess.STDOUT)

up = False
for _ in range(40):
    time.sleep(1)
    if server.poll() is not None:
        break
    with socket.socket() as s:
        if s.connect_ex(("127.0.0.1", 8501)) == 0:
            up = True
            break

if not up:
    print("Streamlit failed to start. Log:")
    print(open("st.log").read()[-2500:])
else:

    try:
        token = re.search(r"[A-Za-z0-9_\-]+", userdata.get("NGROK_AUTHTOKEN")).group(0)
        ngrok.set_auth_token(token)
        tunnel = ngrok.connect(8501)
    except Exception as e:
        print("ngrok failed:", str(e)[:400])
        print("If it says the endpoint is already online, stop the old session in the ngrok dashboard (Agents page) and run this cell again.")
        server.terminate()
    else:
        print("=" * 60)
        print("Open this URL:", tunnel.public_url)
        print("=" * 60)
        print("If ngrok shows a warning page, click 'Visit Site'.")
        print("Keep this cell RUNNING. Press the Stop button to shut the site down.")
        try:
            while True:
                time.sleep(30)
                if server.poll() is not None:
                    print("Streamlit stopped unexpectedly. Log:")
                    print(open("st.log").read()[-2500:])
                    break
        except KeyboardInterrupt:
            pass
        finally:
            ngrok.kill()
            server.terminate()
            print("Site stopped.")